# 🍔 PyQt6 – Lekce 10: Menu, Toolbar a StatusBar

Každá profesionální desktopová aplikace má lištu s menu (Soubor, Úpravy, Nápověda),
panel nástrojů s ikonami a stavový řádek dole.
Tyto prvky jsou součástí `QMainWindow`.

---

## 📦 Přehled komponent

| Komponenta | Třída | Metoda přístupu |
|---|---|---|
| Lišta menu | `QMenuBar` | `self.menuBar()` |
| Jedno menu | `QMenu` | `menubar.addMenu("Soubor")` |
| Položka menu | `QAction` | `menu.addAction(akce)` |
| Panel nástrojů | `QToolBar` | `self.addToolBar(toolbar)` |
| Stavový řádek | `QStatusBar` | `self.statusBar()` |

---

## ⚙️ Co je QAction?

`QAction` je **sdílená akce** – jednou ji definujeme a přidáme jak do menu, tak do toolbaru.
Klávesová zkratka, ikona i text jsou na jednom místě.

```python
akce = QAction("Uložit", self)
akce.setShortcut("Ctrl+S")       # Klávesová zkratka
akce.setStatusTip("Uloží soubor") # Text ve stavovém řádku při hover
akce.triggered.connect(self.ulozit)  # Signál triggered → slot
```

---

## 🟢 Ukázka 1 – Menu Soubor s akcemi

In [3]:
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout,
    QLabel
)
from PyQt6.QtGui import QAction

class AppSMenu(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Ukázka Menu")
        self.resize(450, 200)

        # Central widget
        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)
        self.info = QLabel("Žádná akce zatím nebyla provedena.")
        layout.addWidget(self.info)

        # --- Menu Bar ---
        menubar = self.menuBar()

        # Menu "Soubor"
        menu_soubor = menubar.addMenu("&Soubor")   # & označí klávesovou zkratku Alt+S

        akce_novy = QAction("Nový", self)
        akce_novy.setShortcut("Ctrl+N")
        akce_novy.setStatusTip("Vytvořit nový soubor")
        akce_novy.triggered.connect(lambda: self.akce_info("Nový soubor"))

        akce_otevrit = QAction("Otevřít…", self)
        akce_otevrit.setShortcut("Ctrl+O")
        akce_otevrit.triggered.connect(lambda: self.akce_info("Otevřít soubor"))

        akce_konec = QAction("Konec", self)
        akce_konec.setShortcut("Ctrl+Q")
        akce_konec.triggered.connect(self.close)

        menu_soubor.addAction(akce_novy)
        menu_soubor.addAction(akce_otevrit)
        menu_soubor.addSeparator()          # Čárka oddělující sekce
        menu_soubor.addAction(akce_konec)

        # Menu "Nápověda"
        menu_napoveda = menubar.addMenu("Nápověda")
        akce_o_programu = QAction("O programu", self)
        akce_o_programu.triggered.connect(lambda: self.akce_info("Verze 1.0"))
        menu_napoveda.addAction(akce_o_programu)

        # Status Bar
        self.statusBar().showMessage("Připraven")

    def akce_info(self, text):
        self.info.setText(f"Provedeno: {text}")
        self.statusBar().showMessage(text)


app = QApplication.instance() or QApplication([])
okno = AppSMenu()
okno.show()
app.exec()


0

---

## 🟢 Ukázka 2 – Toolbar s tlačítky

`QToolBar` zobrazuje `QAction` jako tlačítka (s textem nebo ikonami).
Stejnou akci přidáme do menu i do toolbaru – synchronizují se automaticky.

In [4]:
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout,
    QLabel, QToolBar
)
from PyQt6.QtGui import QAction
from PyQt6.QtCore import Qt

class AppSToolbarem(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Toolbar ukázka")
        self.resize(500, 200)

        central = QWidget()
        self.setCentralWidget(central)
        self.info = QLabel("Použijte menu nebo toolbar.", central)
        QVBoxLayout(central).addWidget(self.info)

        # --- Sdílené akce ---
        akce_novy   = QAction("📄 Nový",    self)
        akce_ulozit = QAction("💾 Uložit",  self)
        akce_smazat = QAction("🗑️ Smazat",  self)

        akce_novy.setShortcut("Ctrl+N")
        akce_ulozit.setShortcut("Ctrl+S")

        akce_novy.triggered.connect(lambda:   self.info.setText("Nový soubor"))
        akce_ulozit.triggered.connect(lambda: self.info.setText("Uloženo ✅"))
        akce_smazat.triggered.connect(lambda: self.info.setText("Smazáno 🗑️"))

        # --- Menu (stejné akce) ---
        menu = self.menuBar().addMenu("Soubor")
        menu.addAction(akce_novy)
        menu.addAction(akce_ulozit)
        menu.addAction(akce_smazat)

        # --- Toolbar (stejné akce) ---
        toolbar = QToolBar("Hlavní nástrojová lišta")
        toolbar.setToolButtonStyle(Qt.ToolButtonStyle.ToolButtonTextBesideIcon)  # text vedle ikony
        self.addToolBar(toolbar)
        toolbar.addAction(akce_novy)
        toolbar.addAction(akce_ulozit)
        toolbar.addSeparator()
        toolbar.addAction(akce_smazat)

        self.statusBar().showMessage("Tip: Klávesová zkratka Ctrl+S uloží soubor")


app = QApplication.instance() or QApplication([])
okno = AppSToolbarem()
okno.show()
app.exec()


0

---

## 🟡 Ukázka 3 – Kontextové menu (pravé tlačítko myši)

Kontextové menu se zobrazí po kliknutí pravým tlačítkem.
Nastavíme ho metodou `setContextMenuPolicy` a přetížíme `contextMenuEvent`.

In [5]:
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget,
    QVBoxLayout, QLabel, QMenu
)
from PyQt6.QtGui import QAction
from PyQt6.QtCore import Qt

class OknoSKontextem(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Pravé tlačítko = kontextové menu")
        self.resize(400, 150)

        central = QWidget()
        self.setCentralWidget(central)
        central.setContextMenuPolicy(Qt.ContextMenuPolicy.CustomContextMenu)
        central.customContextMenuRequested.connect(self.zobraz_menu)

        layout = QVBoxLayout(central)
        self.info = QLabel("Klikni pravým tlačítkem kdekoli v okně.")
        layout.addWidget(self.info)

    def zobraz_menu(self, pozice):
        menu = QMenu(self)

        akce_kopiruj = QAction("📋 Kopírovat", self)
        akce_vloz    = QAction("📌 Vložit",    self)
        akce_smazat  = QAction("🗑️ Smazat",   self)

        akce_kopiruj.triggered.connect(lambda: self.info.setText("Kopírováno!"))
        akce_vloz.triggered.connect(lambda:    self.info.setText("Vloženo!"))
        akce_smazat.triggered.connect(lambda:  self.info.setText("Smazáno!"))

        menu.addAction(akce_kopiruj)
        menu.addAction(akce_vloz)
        menu.addSeparator()
        menu.addAction(akce_smazat)

        # Zobrazíme menu na místě kliknutí (v globálních souřadnicích)
        menu.exec(self.centralWidget().mapToGlobal(pozice))


app = QApplication.instance() or QApplication([])
okno = OknoSKontextem()
okno.show()
app.exec()


0

---

## 📋 Shrnutí lekce

| Prvek | Jak vytvořit | Kdy použít |
|---|---|---|
| `QMenuBar` | `self.menuBar()` | Hlavní menu (Soubor, Úpravy…) |
| `QMenu` | `menubar.addMenu("text")` | Podmenu pod jednou položkou |
| `QAction` | `QAction("text", self)` | Sdílená akce (menu + toolbar) |
| `QToolBar` | `self.addToolBar(tb)` | Rychlé akce s ikonami |
| `QStatusBar` | `self.statusBar()` | Info dole – stav, nápověda |
| Kontextové menu | `customContextMenuRequested` | Pravé tlačítko myši |

## ✏️ Úkoly

1. Vytvořte aplikaci s menu **Soubor** (Nový, Otevřít, Uložit, Konec) a menu **Úpravy** (Kopírovat, Vložit, Smazat). Ke každé položce přidejte klávesovou zkratku.
2. Přidejte `QToolBar` se třemi nejdůležitějšími akcemi.
3. Po kliknutí na libovolnou akci zobrazte zprávu ve `statusBar()`.